In [2]:
import pandas as pd
import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as colors
import cmasher as cmr
###import libraries

In [3]:
import sys
sys.path.append('/home/vadilloj/MAP2023/Vadillo-Justice-League-Code')
from Runnable_Modules import Base as base, IonUtils as ions, TrackingUtils as tracking
simulations  = base.get_chars("sims")##import modules previously run

In [21]:
##temporary reload! this is helpful if you are modifying the underlying custom modules while also working on this code and
##want to avoid fully restarting the kernel when you hit a roadblock
import importlib
importlib.reload(base)
importlib.reload(tracking)

<module 'Runnable_Modules.TrackingUtils' from '/home/vadilloj/MAP2023/Vadillo-Justice-League-Code/Runnable_Modules/TrackingUtils.py'>

In [5]:
def get_rbins():
    nbins = 100
    start = 0.1
    end = edge
    edges, binwidth = np.linspace(start, end, num = (nbins+1), endpoint = True, retstep = True)#edges of bins and their widths
    r_bins = edges[0:-1] + (binwidth/2)#midpoints of bins and their widths
    return edges,r_bins

In [6]:
def getHaloChars(h1, subsims,edge):
    """
    function designed to get the halo charachteristics (ie the mass fractions and total mass) for the galaxies,and save it to a datafile
    only should need to be run once! 

    parameters
    h1: (subsim) the central halo ie, the h1 of the simulation you are looking at
    subsims: (dict of subsims)  a dictionary containing all of the subsims of the tracked particles in that halo, should be returned by "tracking.findhaloparticles()"
    edge: (int) an integer representing the edge of the central halo, often just np.max(h1['r'])

    returns 
    HaloChars:  a PD dataframe, where each cell contains an array of the profile of a specific property varying across radius. 
            Indexes denote the specific subhalos that these properties vary for while columns denote the properties themselves.
    """
    ########## parameters for halo charachteristics

    edges,r_bins = get_rbins()
    nbins = len(r_bins)
    mass_keys = ['mass','HI_mass', 'OVI_mass']
    mass_name = ['Mass','HI', 'OVI']
    ################## constant plot Charachteristics defined from the above
    bin_areas = 4/3*np.pi*(edges[1:]**3 - edges[0:-1]**3) ##calculate the area in each bin,by finding the difference in the volume of spheres enclosing its edges. 
    ########

    Chars = {}

    for i in range(len(mass_keys)):
        mass_key = mass_keys[i]
        name = mass_name[i]
        Ratios, Densities = {}, {}
        total_hist, bin_edges = np.histogram(h1.g['r'], bins = edges, weights = h1.g[mass_key])#make histogram for total mass
        
        for halo in subsims:
            #for each satelite, calculate the ratio of total mass, in a given virial bin and its density
            subsnap = subsims[halo]
            halo_hist, bin_edges = np.histogram(subsnap['r'], bins = edges, weights = subsnap[mass_key])
            Ratios[halo] = halo_hist/total_hist
            Densities[halo] = halo_hist/bin_areas
            
        Densities['total'] = total_hist/bin_areas
        Ratios['total'] = np.ones(nbins)
        Chars[name + "_Density"] = Densities
        Chars[name + "_Ratios"] = Ratios
        
    
    indexes = list(Chars['Mass_Ratios'].keys())
    indexes.insert(0, indexes.pop(indexes.index('local')))    

    HaloChars = pd.DataFrame(Chars, index = indexes)##data frame to store halo data
    return HaloChars 

In [7]:
def makeStackplot(sHaloChars, value , ax,rbins, cmapN, local_color = 'black',grouped_color = 'white'):
    values, labels, grouped = group_smalls(sHaloChars, value, len(rbins))
    nHalos  = len(labels)
    
    cmap = mpl.colormaps[cmapN]
    segmentedCmap =cmap(np.linspace(0.35,1, nHalos))
    segmentedCmap[0] = colors.to_rgba(local_color)
    segmentedCmap[-1] = colors.to_rgba(grouped_color)
        
    ax.stackplot(rbins, values, labels = labels, colors = segmentedCmap)

In [8]:
simulations

,gal num,filepath,Halo keys,mass,mstar,mgas,Rvir,nSats,HI_mass,OVI_mass,sat_mass_frac,sat_HI_frac,sat_OVI_frac
Sandra,h148,/home/vadilloj/MAP2023/Sims/h148.cosmo50PLK.30...,"[np.str_('h148_2'), np.str_('h148_3'), np.str_...",2.052077e+12,1.917510e+11,1.527100e+11,266.215150,16.0,1.414185e+10,2.609339e+06,0.009316,0.337959,0.098410
Ruth,h229,/home/vadilloj/MAP2023/Sims/h229.cosmo50PLK.30...,"[np.str_('h229_14'), np.str_('h229_18'), np.st...",1.052426e+12,1.020237e+11,7.695528e+10,214.442995,5.0,1.862303e+10,1.403408e+06,0.000414,0.006439,0.006718
Sonia,h242,/home/vadilloj/MAP2023/Sims/h242.cosmo50PLK.30...,"[np.str_('h242_8'), np.str_('h242_10'), np.str...",1.150238e+12,8.973881e+10,9.158841e+10,275.603382,7.0,2.162918e+10,1.155743e+06,0.007511,0.112475,0.085670
Elena,h329,/home/vadilloj/MAP2023/Sims/h329.cosmo50PLK.30...,"[np.str_('h329_7'), np.str_('h329_29'), np.str...",7.104449e+11,8.991782e+10,2.673112e+10,188.010756,3.0,6.212192e+08,7.602241e+05,0.000727,0.114009,0.004595
Sandra_h,h148,/home/vadilloj/MAP2023/Sims/h148.cosmo50PLK.61...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
def group_smalls(sHaloChars, value,nbins):
    ion = value.split("_")[0]
    grouped = False
    small_values = np.zeros(nbins)
    values, labels, fractions = [], [], []
    halos_grouped = 0
    
    for subsim_label in list(sHaloChars.index):
        subsim_frac = sHaloChars[ion +"_Ratios"][subsim_label]
        profile_values = sHaloChars[value][subsim_label]
        if np.max(subsim_frac)<0.05:
            small_values += profile_values
            grouped = True
            halos_grouped+=1
        else:
            values.append(profile_values)
            labels.append(subsim_label)
            fractions.append(subsim_frac)
    
    small_label = f"<0.5 ({halos_grouped})"
    while(len(labels) > 6):
        index = np.argmin(np.sum(fractions, axis = 1))
        small_values += values.pop(index) 
        labels.pop(index)
        fractions.pop(index)
        halos_grouped+=1
        small_label = f"smallest {halos_grouped}"
    
    if grouped:
        values.append(small_values)
        labels.append(small_label)
    return values, labels, grouped
                

In [10]:
def make_all_stacks(sHaloChars, filename,edge, group_smalls = False):
    plots_filepath  =  '/home/vadilloj/MAP2023/Vadillo-Justice-League-Code/MassFracs-Divided-By-Source-c/Plots/'
    edges,r_bins = get_rbins()
    ######################
    values = ["Density", "Ratios"]

    ions = ["Mass", "HI", "OVI"]
    cmaps = ['cmr.neutral', 'cmr.amethyst', 'cmr.ember']
    localC = ['black','Indigo', 'maroon']
    GroupedColors= ['burlywood', 'skyblue', 'lightcoral']
    yranges = [(10**1, 10**7), (10**-3.5, 10**3.5), (10**-3, 10**-1)]
   ##############
    fig, axs = plt.subplots(len(ions),len(values), figsize = (11,13))
    for row in (range(len(ions))):
        for col in (range(len(values))):
            ax = axs[row, col]
            Hcolumn = ions[row] + '_' + values[col]
            makeStackplot(sHaloChars, Hcolumn, ax, r_bins,
                          cmapN = cmaps[row],local_color = localC[row],grouped_color =GroupedColors[row])
            
            ax.set_xlim(1,edge)
    
            if (col == 0):
                ax.set_yscale("log")
                ax.set_title(ions[row] + " Log Density(Msol/kpc**3) Profile  divided by source")
                ax.set_ylabel('Log '+ ions[row]+ ' mass')
                ax.legend(loc = 'lower left')
                median = np.median(sHaloChars[Hcolumn]['local'])
                ax.set_ylim(yranges[row])
                

                    
                    
            else:
                ax.axhline(1,color = 'black', linestyle = '--')
                ax.set_title("fraction of total " +ions[row] +" divided by source")
                ax.set_ylabel('fraction of '+ ions[row]+ ' mass')
                ax.legend(loc = 'lower left')
                
    
    
    fig.supxlabel('radius (kpc)')
    name = filename + ' (all)Mass Divided by Source'
    fig.suptitle(name, fontsize = 18)
    fig.tight_layout()
    fig.subplots_adjust(top=0.93)
    fig.savefig(plots_filepath+ name + '.png')


subsims = tracking.find_halo_particles(h1, simulations, filename, groupSmalls = group, h = h)

In [11]:
directory_path = '/home/vadilloj/MAP2023/Vadillo-Justice-League-Code/'
Hchars_path = '/home/vadilloj/MAP2023/Vadillo-Justice-League-Code/All_halo_charachteristics.csv'
HChars = pd.read_csv(Hchars_path, index_col = 0)

In [12]:
def readInData(filename):
    import ast
    def string_to_list(x):
        if isinstance(x, str):
            # Try to detect list-like pattern: [ ... ]
            if x.strip().startswith("[") and x.strip().endswith("]"):
                # Replace spaces with commas inside brackets
                inside = x.strip()[1:-1]
                fixed = "[" + ",".join(inside.split()) + "]"
                try:
                    return ast.literal_eval(fixed)
                except (SyntaxError, ValueError):
                    return x
        return x

        
    data_name = data_filepath + filename +"Halo Charachteristics.csv"
    
    HaloChars  = pd.read_csv(data_name, index_col = 0)
    HaloChars  = HaloChars.map(string_to_list)
    return(HaloChars)

    

In [14]:
data_filepath  = '/home/vadilloj/MAP2023/Vadillo-Justice-League-Code/MassFracs-Divided-By-Source-c/Data/'
# filenames  = [  'Ruth','Sonia',  'Elena', 'Sandra']
filenames = ['Sandra_h']

In [17]:

readin = False
for filename in filenames:
   
    data_name = data_filepath + filename +"Halo Charachteristics.csv"
    if readin:
        edge = int(simulations.at[filename, 'Rvir'])
        HaloChars = pd.read_csv(data_name, index_col = 0)
        HaloChars = base.FormatDF(HaloChars)
    else:
        s, h1, h = base.load_in_sim(filename, return_h = True)
        edge = np.max(h1.g['r'])
        ions.calculate_gas_mass(s)
        subsims = tracking.find_halo_particles(h1, filename)##returns a dictionary of subsims for all halos in the simulations, 
        HaloChars = getHaloChars(h1, subsims,edge)
        HaloChars.to_csv(data_name)
    try:
        sHaloChars = HaloChars.drop(['halos', 'total'])
    except KeyError:
        sHaloChars = HaloChars
        pass
    make_all_stacks(sHaloChars, filename,edge )

pynbody.halo : Unable to load AHF substructure file; continuing without. To expose the underlying problem as an exception, pass ignore_missing_substructure=False to the AHFCatalogue constructor


PermissionError: [Errno 13] Unable to synchronously open file (unable to open file: name = '/home/christenc/Data/Sims/tracked_particles_mint.hdf5', errno = 13, error message = 'Permission denied', flags = 0, o_flags = 0)

In [23]:
subsims = tracking.find_halo_particles(h1, filename)##returns a dictionary of subsims for all halos in the simulations, 
HaloChars = getHaloChars(h1, subsims,edge)
HaloChars.to_csv(data_name)
try:
    sHaloChars = HaloChars.drop(['halos', 'total'])
except KeyError:
    sHaloChars = HaloChars
    pass
make_all_stacks(sHaloChars, filename,edge )

PermissionError: [Errno 13] Unable to synchronously open file (unable to open file: name = '/home/christenc/Data/Sims/tracked_particles_mint.hdf5', errno = 13, error message = 'Permission denied', flags = 0, o_flags = 0)

In [13]:
OVI_cmap = mpl.colormaps['cmr.ember'](np.linspace(0.3,0.8, len(sims)))
HI_cmap = mpl.colormaps['cmr.amethyst'](np.linspace(0.05,0.65, len(sims)))

NameError: name 'sims' is not defined

In [ ]:
def readInData(filename):
    path = data_filepath + filename +"Halo Charachteristics.csv"
    HaloChars = pd.read_csv(path, index_col = 0)
    HaloChars = base.FormatDF(HaloChars)
    return HaloChars

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(15,6))
ions = ['HI', 'OVI']        
sims = ['Sandra', 'Ruth','Sonia', 'Elena']
all_colors = [HI_cmap, OVI_cmap]

radii =np.linspace(0,1,100)
for col, ion in enumerate(ions):
    ax = axs[col]
    pltColors = all_colors[col]
    value = ion+"_Ratios"
    for index, filename in enumerate(sims):
        HaloChars= readInData(filename)

        fractions = HaloChars[value]['local']

        if index == 0:
            ax.fill_between(radii, fractions,  label=(filename),ls = '-', color=pltColors[index], alpha = 0.2)
        else:
            ax.plot(radii, fractions, label=(filename), ls = '-', color=pltColors[index])

        
    ax.set_title(ion)

    ax.set_xticks(np.arange(0, 1, 0.1))
    ax.set_xlabel('r/Rvir')
    ax.set_ylabel('Fraction of total mass')
    ax.legend()     
    ax.xaxis.grid(True, which='major', linestyle = "--")
    ax.set_ylim(0,1.01)
    ax.set_xlim(0,1)
    

fig.suptitle('Local Mass Fractions Across Central Halos', fontsize = 14)

    

# plotname = "HI vs OVI Covering Fractions for " + filename +".png"
# fig.savefig(folder_path + '/CF_plots/'+ plotname)
plotname = "HI vs OVI Fractions for all sims.png"
fig.savefig(plotname)


In [ ]:
fig, axs = plt.subplots(2,1,figsize=(6,10))
ions = ['HI', 'OVI']        
sims = ['Sandra', 'Ruth','Sonia', 'Elena']
OVI_cmap = mpl.colormaps['cmr.ember'](np.linspace(0.3,0.8, len(sims)))
HI_cmap = mpl.colormaps['cmr.amethyst'](np.linspace(0.2,0.7, len(sims)))
all_colors = [HI_cmap, OVI_cmap]


radii =np.linspace(0,1,100)
for col, ion in enumerate(ions):
    ax = axs[col]
    pltColors = all_colors[col]
    value = ion+"_Density"
    for index, filename in enumerate(sims):
        HaloChars= readInData(filename)

        total = HaloChars[value]['total']
        local = HaloChars[value]['local']


        plot = ax.fill_between(radii, total, local,  label=(filename),ls = '-', color=pltColors[index], alpha = 0.6)


    ax.set_yscale('log')
    ax.set_title(ion)
    ax.set_xticks(np.arange(0, 1, 0.1))
    ax.set_xlabel('$r / r_{vir}$')
    ax.set_ylabel('Density ($M_{sol} kpc^{-3}$)')
    ax.legend()     
    ax.xaxis.grid(True, which='major', linestyle = "--")
    ax.set_xlim(0,1)
    

fig.suptitle('Local Mass Fractions Across Central Halos', fontsize = 14)
axs[0].axhline(10**-3,color = 'orange', linestyle = '--')
axs[0].axhline(10**4,color = 'orange', linestyle = '--')
axs[1].axhline(10**-3,color = 'orange', linestyle = '--')
axs[1].axhline(10**4,color = 'orange', linestyle = '--')
plt.tight_layout()
plotname = "HI vs OVI Densities for all sims.png"
fig.savefig(plotname)


In [23]:
sHaloChars

,Mass_Density,Mass_Ratios,HI_Density,HI_Ratios,OVI_Density,OVI_Ratios
local,"[10095538.4, 3118179.76, 2343652.92, 2107722.1...","[0.92514617, 0.94188769, 0.95008434, 0.9554776...","[3143689.38, 1189462.77, 949992.629, 899170.22...","[0.923228369, 0.940003591, 0.947843526, 0.9544...","[1592.41027, 401.692004, 248.664722, 182.42060...","[0.9157784, 0.94468631, 0.9492999, 0.95300231,..."
h2,"[3334.84431, 775.124361, 636.027294, 500.21513...","[0.000305602172, 0.000234136628, 0.00025783662...","[862.034413, 282.791702, 308.651642, 228.64211...","[0.000253159434, 0.000223483426, 0.0003079534,...","[0.306225596, 0.0558300501, 0.0880003489, 0.00...","[0.000176107119, 0.000131299312, 0.00033594923..."
h3,"[139484.515, 38388.0523, 25301.7806, 18999.673...","[0.01278224, 0.01159562, 0.01025699, 0.0086129...","[47169.536, 15343.9938, 11219.4937, 8433.30563...","[0.0138525944, 0.0121259863, 0.0111941126, 0.0...","[28.2205322, 4.5741319, 2.8699202, 2.00724228,...","[0.01622933, 0.0107573, 0.01095618, 0.01048624..."
h4,"[0.0, 0.0, 0.0, 0.0, 5.73783468, 2.71615976, 0...","[0.0, 0.0, 0.0, 0.0, 3.81677309e-06, 8.9465059...","[0.0, 0.0, 0.0, 0.0, 2.41006736, 1.73676691, 0...","[0.0, 0.0, 0.0, 0.0, 3.57855799e-06, 1.7604719...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
h6,"[120639.871, 7119.67409, 1077.69921, 452.66460...","[0.01105533, 0.00215059, 0.00043688, 0.0002052...","[39004.9027, 2853.9717, 396.418367, 171.361683...","[0.0114548317, 0.00225542464, 0.000395521577, ...","[16.8025229, 0.606967581, 0.0475095016, 0.0564...","[0.00966295419, 0.00142744679, 0.000181371787,..."
h7,"[54014.5079, 8416.76366, 4375.34806, 3471.3279...","[0.00494984, 0.0025424, 0.00177371, 0.00157363...","[15942.233, 3408.85792, 1892.46199, 1436.2549,...","[0.00468186261, 0.00269393777, 0.00188818081, ...","[11.822448, 0.977384595, 0.600953863, 0.235315...","[0.00679897, 0.00229858, 0.0022942, 0.00122934..."
h10,"[132502.133, 21796.4169, 11684.0781, 8123.5625...","[0.01214238, 0.0065839, 0.00473656, 0.00368259...","[41112.3573, 8473.4466, 5009.20156, 3582.10265...","[0.012073742, 0.00669635941, 0.00499786958, 0....","[17.3338034, 2.71092763, 1.14858357, 0.9627877...","[0.00996849, 0.00637547, 0.00438482, 0.0050298..."
h12,"[20760.6093, 6852.47409, 4363.74437, 4447.6884...","[0.00190248, 0.00206988, 0.001769, 0.00201624,...","[6878.87339, 2942.82986, 1824.84082, 1858.5985...","[0.00202016494, 0.00232564709, 0.00182071261, ...","[2.52127501, 0.772693628, 0.298603, 0.44875144...","[0.00144996, 0.0018172, 0.00113994, 0.00234437..."
h23,"[173098.752, 56428.8059, 37355.6641, 30208.824...","[0.01586262, 0.01704507, 0.01514347, 0.0136943...","[54346.4344, 22136.031, 15981.8484, 13391.5718...","[0.0159602823, 0.0174935686, 0.0159456938, 0.0...","[28.7557791, 7.50188225, 4.30333411, 2.7786612...","[0.01653715, 0.01764268, 0.01642836, 0.0145162..."
h27,"[107133.601, 32023.6664, 24302.1659, 18862.641...","[0.00981763, 0.00967317, 0.00985176, 0.0085508...","[36195.5501, 12723.6753, 10140.5526, 8283.8182...","[0.0106297902, 0.0100552121, 0.0101176123, 0.0...","[23.7192249, 4.88733259, 2.51090017, 1.5620371...","[0.01364068, 0.01149387, 0.00958559, 0.0081604..."


In [24]:
subsims = tracking.find_halo_particles(h1, simulations, filename, groupSmalls = True)##returns a dictionary of subsims for all halos in the simulations, 

NameError: name 'h1' is not defined